In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# --- Step 1: Read the .eol file ---
file_path = "NSF_NCAR_EOL/ebsmi503707/D20140912_042945_P.QC.eol"

with open(file_path, 'r') as f:
    lines = f.readlines()

# Data starts after the line with column names
for i, line in enumerate(lines):
    if "Time" in line and "Press" in line:
        header_line_idx = i
        break

# Read data into DataFrame
df = pd.read_csv(
    file_path,
    delim_whitespace=True,
    skiprows=header_line_idx+2,
    names=["Time_sec","UTC_hh","UTC_mm","UTC_ss","Press_mb","Temp_C","Dewpt_C",
           "RH","Uwind","Vwind","Wspd","Dir","dZ","GeoPoAlt","Lon","Lat","GPSAlt"]
)

# --- Step 2: Convert numeric columns safely ---
cols_to_numeric = ["Press_mb", "Temp_C", "Dewpt_C", "RH", "Uwind", "Vwind", "Wspd"]
for col in cols_to_numeric:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# --- Step 3: Replace missing value flags (999, -999) with NaN ---
df.replace([999, -999], np.nan, inplace=True)

# --- Step 4: Interpolate missing values ---
df.interpolate(method='linear', inplace=True)

# Drop rows where essential data is still missing
df = df.dropna(subset=["Press_mb", "Temp_C", "Uwind", "Vwind"], how="any")

# --- Step 5: Compute wind speed if missing ---
df["WindSpeed"] = (df["Uwind"]**2 + df["Vwind"]**2)**0.5

# --- Step 6: Plot ---
fig, ax1 = plt.subplots(figsize=(6,8))

# Temperature vs Pressure
ax1.plot(df["Temp_C"], df["Press_mb"], color='red', label="Temp (°C)")
ax1.set_xlabel("Temperature (°C) / Wind Speed (m/s)")
ax1.set_ylabel("Pressure (mb)")
ax1.invert_yaxis()  # Pressure decreases upward
ax1.grid(True)

# Wind speed vs Pressure on same x-axis
ax1.plot(df["WindSpeed"], df["Press_mb"], color='blue', label="Wind Speed (m/s)")

# Legend
ax1.legend()

plt.title("HS3 2014 AVAPS Dropsonde: Temp & Wind vs Pressure (Interpolated)")
plt.show()


In [ ]:
import os
import pandas as pd
import numpy as np
from geopy.distance import great_circle
import glob

# --- Path to your EOL dropsonde files ---
eol_files = sorted(glob.glob("NSF_NCAR_EOL/ebsmi503707/*.eol"))

# --- Storm center coordinates (replace with actual) ---
storm_center_lat = 18.0
storm_center_lon = -52.0

# --- Prepare a list to store results ---
rmw_list = []

# --- Loop through all .eol files ---
for file_path in eol_files:
    
    fname = os.path.basename(file_path)  # just the filename
    
    with open(file_path, "r") as f:
        lines = f.readlines()
    
    # Find where the data starts (skip headers)
    for i, line in enumerate(lines):
        if line.strip().startswith("PRES") or line.strip().startswith("PRES (hPa)"):
            header_idx = i
            break
    
    # Read the data into pandas
    data = pd.read_csv(
        file_path,
        skiprows=header_idx+1,
        delim_whitespace=True,
        names=["PRES", "TEMP", "RH", "UWND", "VWND", "HGHT"],
        comment="#"
    )
    
    # Compute wind speed
    data["WSPD"] = np.sqrt(data["UWND"]**2 + data["VWND"]**2)
    
    # Get sonde lat/lon from header
    lat_line = [l for l in lines if "Latitude" in l][0]
    lon_line = [l for l in lines if "Longitude" in l][0]
    sonde_lat = float(lat_line.split(":")[1].strip())
    sonde_lon = float(lon_line.split(":")[1].strip())
    
    # Compute distance from storm center (same distance for all points)
    data["RADIUS"] = data.apply(lambda row: great_circle(
        (storm_center_lat, storm_center_lon),
        (sonde_lat, sonde_lon)
    ).km, axis=1)
    
    # Find max wind and its radius
    idx_max = data["WSPD"].idxmax()
    max_wind = data.loc[idx_max, "WSPD"]
    rmw = data.loc[idx_max, "RADIUS"]
    
    # Store results
    rmw_list.append({
        "file": fname,
        "max_wind_mps": max_wind,
        "RMW_km": rmw
    })

# --- Ensure output directory exists ---
output_dir = "NSF_NCAR_EOL/ebsmi503707"
os.makedirs(output_dir, exist_ok=True)

# --- Save results to CSV ---
output_csv = os.path.join(output_dir, "rmw_results.csv")
rmw_df = pd.DataFrame(rmw_list)
rmw_df.to_csv(output_csv, index=False)
print(f"RMW results saved to {output_csv}")
